# Sea surface salinity, mixed layer depth and the AMOC

Annual-mean, area-averaged time series of sea surface salinity (SSS) and mixed layer depth (MLD) in the two main North Atlantic deep-water formation regions (Labrador and GIN Seas), and the AMOC strength at 26°N and 55°N, comparing ACCESS-OM3 with ACCESS-OM2 (`025deg_jra55_iaf_omip2_cycle1`). A WOA23 climatological SSS and the RAPID AMOC estimate are shown for reference.

Context: low-AMOC bias, see [#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195) and [#196](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/196). Fresh surface water in the convection regions suppresses deep mixing and weakens the AMOC.

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import intake
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from distributed import Client
from exptdata_access import get_exptname_from_path

client = Client(threads_per_worker=1)
om3exptname = get_exptname_from_path(esm_file)
om2exptname = "025deg_jra55_iaf_omip2_cycle1"


## Regions

Boxes in true longitude/latitude, applied to each model's 2D `geolon`/`geolat` (both regions lie partly north of 65°N, where the tripolar grid's nominal 1D coordinates are not true longitude/latitude).

- **Labrador Sea**: 53–65°N, 65–43°W
- **GIN Seas** (Greenland–Iceland–Norwegian): 65–80°N, 20°W–15°E

In [ ]:
regions = {  # lon_min, lon_max, lat_min, lat_max (degrees east / north)
    "Labrador Sea": (-65, -43, 53, 65),
    "GIN Seas": (-20, 15, 65, 80),
}

def region_mask(lon, lat, box):
    lon = (lon + 180) % 360 - 180  # wrap to [-180, 180) so boxes work on any grid convention
    lon_min, lon_max, lat_min, lat_max = box
    return (lon >= lon_min) & (lon <= lon_max) & (lat >= lat_min) & (lat <= lat_max)

## Load ACCESS-OM3 monthly SSS and MLD (`mlotst`)

The OM3 datastore has no monthly-mean `sos`, so SSS is taken as the top model level (0.5 m) of monthly `so`.

In [ ]:
om3cat = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=["variable", "variable_long_name", "variable_standard_name",
                            "variable_cell_methods", "variable_units"],
)

def load_om3(variable):
    return om3cat.search(variable=variable, frequency="1mon").to_dask(
        xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable]

om3 = {"SSS": load_om3("so").isel(z_l=0), "MLD": load_om3("mlotst")}

om3area = xr.open_dataset(om3cat.search(variable="areacello", frequency="fx").df.path.iloc[0])["areacello"]
# geolon/geolat from the geometry file: the static file's copies are NaN over land-only processors
om3geom = xr.open_dataset(om3cat.search(variable="geolon", filename="access-om3.mom6.geometry.nc").df.path.iloc[0])
om3geom = om3geom.rename({"lonh": "xh", "lath": "yh"}).assign_coords(xh=om3area.xh, yh=om3area.yh)
om3lon, om3lat = om3geom["geolon"], om3geom["geolat"]

## Load ACCESS-OM2 monthly SSS (`sss`) and MLD (`mld`)

In [ ]:
om2cat = intake.cat.access_nri[om2exptname]

def load_om2(variable):
    return om2cat.search(variable=variable, frequency="1mon").to_dask(
        xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable]

om2 = {"SSS": load_om2("sss"), "MLD": load_om2("mld")}

# reference grid file: the run's own ocean_grid.nc has NaN geolon/geolat over land-only processors
om2grid = xr.open_dataset("/g/data/ik11/grids/ocean_grid_025.nc")
om2area = om2grid["area_t"]
om2lon, om2lat = om2grid["geolon_t"], om2grid["geolat_t"]

Check the regions on each model grid (land is white):

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5),
                        subplot_kw=dict(projection=ccrs.PlateCarree()))
for ax, (name, mld, lon, lat) in zip(axs, [("ACCESS-OM3", om3["MLD"], om3lon, om3lat),
                                          ("ACCESS-OM2", om2["MLD"], om2lon, om2lat)]):
    ocean = mld.isel(time=0).notnull().load()
    ax.pcolormesh(lon, lat, ocean.where(ocean), cmap="Greys", vmin=0, vmax=4,
                  transform=ccrs.PlateCarree())
    for i, (region, box) in enumerate(regions.items()):
        m = region_mask(lon, lat, box) & ocean
        ax.pcolormesh(lon, lat, m.where(m), cmap=["Blues", "Oranges"][i], vmin=0, vmax=1.5,
                      transform=ccrs.PlateCarree())
    ax.set_extent([-80, 30, 45, 82], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="110m", linewidth=0.5)
    ax.set_title(name)
plt.show()

## Area-weighted regional means, then annual means

Annual means weight each month by its length.

In [ ]:
def regional_annual_mean(da, area, lon, lat, box):
    inside = region_mask(lon, lat, box).values
    rows, cols = inside.any(axis=1).nonzero()[0], inside.any(axis=0).nonzero()[0]
    subset = {area.dims[0]: slice(rows[0], rows[-1] + 1), area.dims[1]: slice(cols[0], cols[-1] + 1)}
    da, area, lon, lat = (x.isel(subset) for x in (da, area, lon, lat))  # only read the region's bounding box
    weights = area.where(region_mask(lon, lat, box) & da.isel(time=0).notnull(), 0.0)
    monthly = da.weighted(weights).mean(list(area.dims))
    days = monthly.time.dt.days_in_month
    return ((monthly * days).groupby("time.year").sum() / days.groupby("time.year").sum()).load()

timeseries = {
    var: {
        region: {
            "ACCESS-OM3": regional_annual_mean(om3[var], om3area, om3lon, om3lat, box),
            "ACCESS-OM2": regional_annual_mean(om2[var], om2area, om2lon, om2lat, box),
        }
        for region, box in regions.items()
    }
    for var in ["SSS", "MLD"]
}

## WOA23 climatological SSS for reference

WOA23 monthly climatology already interpolated onto the ACCESS-OM3 25 km grid (used for the model initial conditions); we average the 12 months at the top level.

In [ ]:
woa23_sss = (
    xr.open_mfdataset("/g/data/ik11/inputs/access-om3/woa23/025/2026.06.11/woa23_ts_*_mom.nc")["salt"]
    .isel(depth=0).mean("time")
    .rename({"lon": "xh", "lat": "yh"})
)
woa23_sss = woa23_sss.assign_coords(xh=om3area.xh, yh=om3area.yh)  # same grid as OM3

woa23_ref = {
    region: float(woa23_sss.weighted(om3area.where(region_mask(om3lon, om3lat, box) & woa23_sss.notnull(), 0.0))
                  .mean(["yh", "xh"]))
    for region, box in regions.items()
}
woa23_ref

## Sea surface salinity

In [ ]:
colors = {"ACCESS-OM3": "C0", "ACCESS-OM2": "C1"}

fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, region in zip(axs, regions):
    for model, ts in timeseries["SSS"][region].items():
        ts.plot(ax=ax, color=colors[model], label=model)
    ax.axhline(woa23_ref[region], color="k", ls="--", lw=1, label="WOA23 climatology")
    ax.set_title(f"{region} SSS")
    ax.set_xlabel("Year")
    ax.set_ylabel("SSS (psu)")
axs[0].legend()
plt.tight_layout()
mkmd.savefig(fig, "Labrador and GIN Seas SSS",
             f"Annual-mean, area-averaged sea surface salinity in the Labrador Sea (53-65°N, 65-43°W) and GIN Seas "
             f"(65-80°N, 20°W-15°E): ACCESS-OM3 {om3exptname}, ACCESS-OM2 {om2exptname} and WOA23 climatology (dashed). "
             "GitHub issue: [#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195)", dpi=150)

## Mixed layer depth

Both models diagnose MLD from a 0.03 kg m⁻³ density-difference criterion (OM3 `mlotst`; OM2 `mld`, which uses the equivalent buoyancy criterion). Annual means smooth over the late-winter convective maximum, so they mostly reflect the depth and duration of winter mixing.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, region in zip(axs, regions):
    for model, ts in timeseries["MLD"][region].items():
        ts.plot(ax=ax, color=colors[model], label=model)
    ax.set_title(f"{region} MLD")
    ax.set_xlabel("Year")
    ax.set_ylabel("MLD (m)")
axs[0].legend()
plt.tight_layout()
mkmd.savefig(fig, "Labrador and GIN Seas MLD",
             f"Annual-mean, area-averaged mixed layer depth in the Labrador Sea (53-65°N, 65-43°W) and GIN Seas "
             f"(65-80°N, 20°W-15°E): ACCESS-OM3 {om3exptname} and ACCESS-OM2 {om2exptname}. "
             "GitHub issue: [#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195)", dpi=150)

## AMOC at 26°N and 55°N

Same method as `Overturning_in_ACCESS_OM3.ipynb`: the North Atlantic (81.5°W–10°E) overturning streamfunction in σ₂ density space, accumulated from light to dense, and its maximum over 1035–1038 kg m⁻³. For ACCESS-OM3 this uses `vmo` on σ₂ layers; for ACCESS-OM2 `ty_trans_rho` plus the GM contribution `ty_trans_rho_gm`. 26°N is compared with RAPID (17.2 Sv, McCarthy et al. 2015).

In [ ]:
rho0 = 1035.0  # kg m-3, converts mass transport (kg s-1) to volume transport
north_atlantic = slice(-81.5, 10)
amoc_latitudes = [26, 55]

def annual_mean(da):
    days = da.time.dt.days_in_month
    return (da * days).groupby("time.year").sum() / days.groupby("time.year").sum()

vmo = load_om3("vmo")  # on rho2_l layers
amoc_om3 = {
    lat: annual_mean(
        (vmo.sel(yq=lat, method="nearest").sel(xh=north_atlantic).sum("xh").cumsum("rho2_l") / rho0 / 1e6)
        .sel(rho2_l=slice(1035, 1038)).max("rho2_l")
    ).load()
    for lat in amoc_latitudes
}

ty = load_om2("ty_trans_rho")
ty_gm = load_om2("ty_trans_rho_gm")

def om2_streamfunction(lat):
    row = dict(grid_yu_ocean=lat, method="nearest")
    t = ty.sel(**row).sel(grid_xt_ocean=north_atlantic).sum("grid_xt_ocean")
    gm = ty_gm.sel(**row).sel(grid_xt_ocean=north_atlantic).sum("grid_xt_ocean")
    psi = t.cumsum("potrho") - t.sum("potrho") + gm  # ty_trans_rho_gm is already a streamfunction
    return psi / rho0 / 1e6  # ty_trans_rho is in kg s-1 here

amoc_om2 = {
    lat: annual_mean(om2_streamfunction(lat).sel(potrho=slice(1035, 1038)).max("potrho")).load()
    for lat in amoc_latitudes
}

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, lat in zip(axs, amoc_latitudes):
    amoc_om3[lat].plot(ax=ax, color=colors["ACCESS-OM3"], label="ACCESS-OM3")
    amoc_om2[lat].plot(ax=ax, color=colors["ACCESS-OM2"], label="ACCESS-OM2")
    ax.set_title(f"AMOC at {lat}°N")
    ax.set_xlabel("Year")
    ax.set_ylabel("Overturning (Sv)")
axs[0].axhline(17.2, color="k", ls="--", lw=1, label="RAPID (McCarthy et al. 2015)")
axs[0].legend()
plt.tight_layout()
mkmd.savefig(fig, "AMOC at 26N and 55N",
             f"Annual-mean maximum North Atlantic overturning streamfunction (σ2 1035-1038 kg m-3) at 26°N and 55°N: "
             f"ACCESS-OM3 {om3exptname} and ACCESS-OM2 {om2exptname}; RAPID estimate dashed. "
             "GitHub issues: [#7](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/7), "
             "[#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195)", dpi=150)

In [ ]:
client.close()